In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =====================================================================
# Method roster (0..6). Methods 5 (EAP-TLS Classic) and 6 (EAP-TLS
# Hybrid PQC) are the newly added variants; all numbers below come from
# the latest end-to-end benchmark (analysis_e2e.md / "analisys2.md").
# =====================================================================
methods = ["EAP-EDHOC Method 0", "EAP-EDHOC Method 1", "EAP-EDHOC Method 2", "EAP-EDHOC Method 3",
           "EAP-EDHOC Hybrid", "EAP-TLS Classic", "EAP-TLS Hybrid PQC"]
# Two-line wrapped labels for bar-chart x-axes, to avoid overlap without rotation
methods_wrapped = ["EAP-EDHOC\nMethod 0", "EAP-EDHOC\nMethod 1", "EAP-EDHOC\nMethod 2", "EAP-EDHOC\nMethod 3",
                    "EAP-EDHOC\nHybrid", "EAP-TLS\nClassic", "EAP-TLS\nHybrid PQC"]
# Short labels for the crowded-point summary table (keeps the box compact)
colors7 = ['#64B5F6', '#1565C0', '#9575CD', '#5E35B1', '#E53935', '#43A047', '#FB8C00']
markers7 = ['o', 's', '^', 'D', 'v', 'P', 'X']
loss = [0, 1, 5, 10, 20, 30]

# =====================================================================
# Chart 1 — Handshake time per method: two side-by-side panels.
# LEFT: pure cryptographic computation time (primitives only), Source:
# analysis_e2e.md section 1b "Total komputasi handshake per method" —
# independen terhadap transport (tidak ada overhead RADIUS/TLS-framing).
# RIGHT: real end-to-end wall-clock handshake time, Source: analysis_e2e.md
# section "Pembagian beban initiator vs responder (live, wall-clock nyata)".
# initiator_ms = kerja lokal UE; responder_ms = waktu tunggu balasan yang
# didominasi pemrosesan penuh di sisi FreeRADIUS.
# =====================================================================
compute_initiator = [0.091207, 0.061769, 0.105301, 0.075862, 0.492320, 0.091207, 0.407846]  # handshake-compute.csv (symmetry-no-eph)
compute_responder = [0.091207, 0.105301, 0.061769, 0.075862, 0.492320, 0.091207, 0.367311]

# live EAP-over-RADIUS, benchmark-results/analisys2.md (symmetry-no-eph run)
e2e_initiator = [0.142, 0.123, 0.139, 0.134, 3.511, 0.585, 0.846]
e2e_responder = [0.924, 0.822, 0.812, 1.267, 9.712, 2.114, 8.850]

x = np.arange(len(methods))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))

panel_data = [
    ("Cryptographic Computation Time (Primitives Only)", compute_initiator, compute_responder, "Computation Time (ms)", False),
    ("End-to-End Handshake Time (Live, Wall-Clock)", e2e_initiator, e2e_responder, "Handshake Time (ms)", True),
]

for ax, (subtitle, init_vals, resp_vals, ylabel, show_total) in zip(axes, panel_data):
    b1 = ax.bar(x - width/2, init_vals, width, label='Initiator', color='#1565C0')
    b2 = ax.bar(x + width/2, resp_vals, width, label='Responder', color='#90CAF9')

    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xlabel("EDHOC Authentication Method", fontsize=11)
    ax.set_title(subtitle, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(methods_wrapped, rotation=0, ha='center', fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.legend(frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ymax = max(max(init_vals), max(resp_vals))

    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width()/2,
                h + ymax * 0.015,
                f"{h:.3f}",
                ha='center',
                va='bottom',
                fontsize=8
            )

    if show_total:
        totals = [i + r for i, r in zip(init_vals, resp_vals)]
        for xi, total in zip(x, totals):
            bar_top = max(init_vals[xi], resp_vals[xi])
            ax.text(
                xi,
                bar_top + ymax * 0.07,
                f"Total\n{total:.3f} ms",
                ha='center',
                va='bottom',
                fontsize=8,
                fontweight='bold',
                color='#333333'
            )
        ax.set_ylim(0, ymax * 1.24)
    else:
        ax.set_ylim(0, ymax * 1.15)

fig.suptitle("Handshake Time per Method: Cryptographic Computation vs End-to-End (Live)",
             fontsize=15, fontweight='bold', y=1.05)

plt.tight_layout()
plt.savefig("handshake_computation_grouped.png", dpi=600, bbox_inches='tight')
plt.close()


# =====================================================================
# Chart 2 — Communication cost & MTU fragmentation, two side-by-side
# panels. LEFT: per-message communication cost breakdown (bytes).
# RIGHT: total MTU fragmentation per method (full handshake).
# Source: analysis_e2e.md section 4 "Pengaruh ukuran paket terhadap MTU
# dan fragmentasi" — hasil observasi langsung dari handshake live.
# EAP-EDHOC Hybrid (4) and EAP-TLS Hybrid PQC (6) have 4 messages
# (MSG1-MSG4); EAP-TLS Classic (5) also has 4 messages;
# methods 0-3 have 3 messages (MSG4=0)
# =====================================================================
methods_c2 = [
    "EAP-TLS Hybrid PQC",
    "EAP-EDHOC Hybrid",
    "EAP-TLS Classic",
    "EAP-EDHOC Method 0",
    "EAP-EDHOC Method 2",
    "EAP-EDHOC Method 1",
    "EAP-EDHOC Method 3"
]

MSG1 = [1589, 1141, 289, 34, 34, 34, 34]
MSG2 = [12600, 4880, 1148, 147, 147, 91, 91]
MSG3 = [10983, 4975, 990, 115, 59, 115, 59]
MSG4 = [23, 1232, 51, 0, 0, 0, 0]

colors = [
    "#c6dbef",   # MSG1
    "#6baed6",   # MSG2
    "#2171b5",   # MSG3
    "#08306b"    # MSG4
]

y = np.arange(len(methods_c2))

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(19, 6.5))

# --- Left panel: per-message communication cost breakdown ---
b1 = ax.barh(y, MSG1, color=colors[0], edgecolor='black', linewidth=0.5, label="MSG1")
b2 = ax.barh(y, MSG2, left=np.array(MSG1), color=colors[1], edgecolor='black', linewidth=0.5, label="MSG2")
b3 = ax.barh(y, MSG3, left=np.array(MSG1)+np.array(MSG2), color=colors[2], edgecolor='black', linewidth=0.5, label="MSG3")
b4 = ax.barh(y, MSG4, left=np.array(MSG1)+np.array(MSG2)+np.array(MSG3), color=colors[3], edgecolor='black', linewidth=0.5, label="MSG4")

totals = np.array(MSG1) + np.array(MSG2) + np.array(MSG3) + np.array(MSG4)
x_total = max(totals) + 300

for i, total in enumerate(totals):
    ax.text(x_total, i, f"{total:,} B", fontsize=10, fontweight='bold', va='center')

bars_all = [("MSG1", b1), ("MSG2", b2), ("MSG3", b3), ("MSG4", b4)]

offsets = {
    "MSG1": -0.28,
    "MSG2": -0.08,
    "MSG3":  0.12,
    "MSG4":  0.28
}

for name, bars in bars_all:
    for bar in bars:
        value = bar.get_width()
        if value == 0:
            continue

        x_left = bar.get_x()
        x_center = x_left + value/2
        y_center = bar.get_y() + bar.get_height()/2

        if value < 1300:
            ax.annotate(
                f"{name}: {int(value)}",
                xy=(x_left+value, y_center),
                xytext=(x_left+value+150, y_center+offsets[name]),
                fontsize=8,
                ha='left',
                va='center',
                arrowprops=dict(arrowstyle='-', lw=0.8),
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.9)
            )
        else:
            ax.text(x_center, y_center, str(int(value)), ha='center', va='center', fontsize=9, fontweight='bold')

ax.axvline(totals[0], linestyle='--', color='black', linewidth=1.5,
           label=f'{methods_c2[0]} Total ({totals[0]:,} B)')

ax.set_yticks(y)
ax.set_yticklabels(methods_c2)
ax.set_xlabel("Communication Cost (Bytes)", fontsize=12)
ax.set_title("Per-Message Communication Cost Breakdown", fontsize=13, fontweight='bold', pad=12)
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(ncol=5, bbox_to_anchor=(0.5, -0.14), loc='upper center', frameon=False, fontsize=9)
ax.set_xlim(0, 26000)

# --- Right panel: total MTU fragmentation per method ---
# observed on the live path, analisys2.md section 4
total_eap_frags = [3, 3, 3, 3, 14, 5, 27]
total_ip_frags = [3, 3, 3, 3, 10, 4, 20]

x7 = np.arange(len(methods))
width7 = 0.35

bars7a = ax2.bar(x7 - width7/2, total_eap_frags, width7, label='Total EAP Fragments', color='#1565C0')
bars7b = ax2.bar(x7 + width7/2, total_ip_frags, width7, label='Total IP Fragments', color='#90CAF9')

ax2.set_ylabel("Number of Fragments", fontsize=12)
ax2.set_xlabel("All Variant Authentication Method", fontsize=12)
ax2.set_title("Total MTU Fragmentation per Method (Full Handshake)", fontsize=13, fontweight='bold')
ax2.set_xticks(x7)
ax2.set_xticklabels(methods_wrapped, rotation=0, ha='center', fontsize=9)
ax2.grid(axis='y', linestyle='--', alpha=0.3)
ax2.legend(frameon=False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

for bars in [bars7a, bars7b]:
    for bar in bars:
        h = bar.get_height()
        ax2.text(
            bar.get_x() + bar.get_width()/2,
            h + 0.3,
            f"{int(h)}",
            ha='center',
            va='bottom',
            fontsize=9
        )

ax2.set_ylim(0, max(total_eap_frags) * 1.18)

fig.suptitle("Communication Cost and MTU Fragmentation per Method",
             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig("communication_breakdown_methods.png", dpi=300, bbox_inches='tight')
plt.close()


# =====================================================================
# DATA — Chart 3,4,5,6 versi network-emulated (analysis.md, run
# 2026-08-14T14:21:21). Untuk pertama kalinya section 2 mencakup
# Method 0-6 secara penuh. PENTING: simulasi ini SELALU 2 round-trip
# tetap (tidak memodelkan fragmentasi EAP RFC 3579/5216), sehingga
# TIDAK menangkap efek amplifikasi loss dari pesan besar yang
# sebenarnya terpecah jadi banyak fragmen (paling berpengaruh untuk
# Method 4 & 6). Method 6 di sini terlihat tangguh (95.8% @ 30% loss)
# padahal live sungguhan lewat RADIUS (panel kanan, dengan fragmentasi
# EAP nyata dan 28 round-trip) hanya ~79.2% pada loss yang sama.
# =====================================================================
# sim_* diperbarui dari benchmark-results/lossy-network.csv (run branch symmetry-no-eph, semua method)
sim_mean = {
    0: [4.734, 5.882, 14.991, 24.408, 49.984, 92.935],
    1: [3.635, 5.869, 14.848, 22.587, 44.025, 92.913],
    2: [4.055, 6.696, 13.243, 25.636, 62.391, 91.936],
    3: [4.953, 7.191, 17.149, 24.871, 50.879, 92.859],
    4: [6.323, 6.345, 14.796, 27.498, 54.395, 81.021],
    5: [4.810, 8.048, 13.962, 18.159, 59.973, 94.884],
    6: [4.851, 6.632, 15.213, 26.667, 50.846, 89.130],
}
sim_p95 = {
    0: [9.438, 12.318, 54.405, 88.711, 147.857, 261.683],
    1: [4.899, 11.381, 54.332, 89.863, 134.038, 255.791],
    2: [7.645, 13.481, 51.966, 93.370, 177.804, 226.199],
    3: [11.294, 19.819, 86.103, 92.610, 140.767, 250.958],
    4: [12.843, 14.353, 51.779, 102.028, 138.873, 232.531],
    5: [9.619, 49.024, 49.667, 85.970, 181.839, 260.719],
    6: [10.573, 14.946, 50.582, 98.462, 168.616, 208.975],
}
sim_retx = {
    0: [0.000, 0.025, 0.217, 0.458, 1.008, 2.125],
    1: [0.000, 0.017, 0.217, 0.400, 0.983, 2.192],
    2: [0.000, 0.033, 0.175, 0.467, 1.325, 2.233],
    3: [0.000, 0.042, 0.283, 0.467, 1.083, 2.017],
    4: [0.000, 0.025, 0.242, 0.517, 1.133, 1.775],
    5: [0.000, 0.067, 0.192, 0.325, 1.358, 2.042],
    6: [0.000, 0.025, 0.267, 0.508, 1.042, 2.192],
}
sim_success = {
    0: [100.0, 100.0, 100.0, 100.0, 100.0, 98.3],
    1: [100.0, 100.0, 100.0, 100.0, 99.2, 97.5],
    2: [100.0, 100.0, 100.0, 100.0, 100.0, 96.7],
    3: [100.0, 100.0, 100.0, 100.0, 100.0, 99.2],
    4: [100.0, 100.0, 100.0, 100.0, 100.0, 99.2],
    5: [100.0, 100.0, 100.0, 100.0, 98.3, 100.0],
    6: [100.0, 100.0, 100.0, 100.0, 100.0, 95.8],
}

# =====================================================================
# DATA — Chart 3,4,5,6 versi end-to-end (analysis_e2e.md, live
# EAP-over-RADIUS, symmetry-no-eph run 2026-10-27) — Method 0..6, termasuk
# Method 5 (EAP-TLS Classic, Ed25519) dan Method 6 (EAP-TLS Hybrid PQC),
# section 2 "Performa pada jaringan lossy".
# =====================================================================
e2e_mean = {
    0: [1.141, 3.351, 12.575, 30.902, 69.538, 128.089],
    1: [0.683, 2.871, 14.717, 37.585, 68.764, 115.522],
    2: [1.167, 5.998, 10.403, 33.138, 69.738, 135.643],
    3: [0.930, 4.962, 14.984, 32.980, 72.222, 132.048],
    4: [11.828, 31.874, 88.514, 164.798, 390.278, 598.248],
    5: [2.874, 8.019, 35.999, 69.652, 151.254, 280.789],
    6: [9.863, 34.532, 162.307, 319.445, 667.534, 1206.565],
}
e2e_p95 = {
    0: [1.877, 4.585, 44.806, 130.403, 174.457, 259.997],
    1: [0.994, 2.867, 57.040, 124.858, 170.816, 255.389],
    2: [2.033, 43.185, 45.640, 88.631, 174.240, 316.764],
    3: [1.193, 42.185, 60.235, 123.268, 212.064, 300.150],
    4: [13.325, 105.042, 204.883, 338.988, 652.323, 938.116],
    5: [3.440, 49.500, 141.814, 178.646, 293.311, 551.183],
    6: [11.342, 115.067, 327.416, 615.732, 982.990, 1706.477],
}
e2e_retx = {
    0: [0.000, 0.050, 0.275, 0.667, 1.642, 2.917],
    1: [0.000, 0.042, 0.317, 0.875, 1.625, 2.625],
    2: [0.000, 0.108, 0.217, 0.717, 1.608, 3.067],
    3: [0.000, 0.083, 0.317, 0.717, 1.642, 3.000],
    4: [0.000, 0.317, 1.567, 3.192, 8.100, 11.342],
    5: [0.000, 0.092, 0.650, 1.450, 3.333, 6.108],
    6: [0.000, 0.467, 3.225, 7.008, 14.842, 21.392],
}
e2e_success = {
    0: [100.0, 100.0, 100.0, 100.0, 100.0, 96.7],
    1: [100.0, 100.0, 100.0, 100.0, 100.0, 95.8],
    2: [100.0, 100.0, 100.0, 100.0, 99.2, 96.7],
    3: [100.0, 100.0, 100.0, 100.0, 100.0, 98.3],
    4: [100.0, 100.0, 100.0, 100.0, 98.3, 86.7],
    5: [100.0, 100.0, 100.0, 100.0, 100.0, 96.7],
    6: [100.0, 100.0, 100.0, 100.0, 96.7, 77.5],
}


def fmt_val(v):
    """Adaptive precision so labels stay compact across widely different scales."""
    if v >= 100:
        return f"{v:.0f}"
    elif v >= 10:
        return f"{v:.1f}"
    else:
        return f"{v:.2f}"


# Points that sit too close together (in x AND often in y across the 7
# series) for inline labels to stay readable; these get summarized in a
# small corner table instead of individual annotations.
CROWDED_LOSS = {0, 1}


def build_crowded_table(data):
    """Build a compact monospace table of values at the crowded loss points,
    one row per method (full names, matching the legend exactly), columns
    = crowded loss levels."""
    name_w = max(len(methods[i]) for i in data.keys()) + 1
    header = " " * name_w + "".join(f"{f'{lp}%':>8}" for lp in sorted(CROWDED_LOSS))
    lines = [header]
    for i in sorted(data.keys()):
        row = f"{methods[i]:<{name_w}}"
        for lp in sorted(CROWDED_LOSS):
            li = loss.index(lp)
            row += f"{fmt_val(data[i][li]):>8}"
        lines.append(row)
    return "\n".join(lines)


def plot_line_pair(sim_data, e2e_data, ylabel, suptitle, fname,
                    left_title="Network-Emulated (Analytical)",
                    right_title="End-to-End (Live EAP-over-RADIUS)"):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5.6), sharey=True)

    for ax, data, subtitle in zip(axes, [sim_data, e2e_data], [left_title, right_title]):
        for idx, i in enumerate(sorted(data.keys())):
            ax.plot(loss, data[i], marker=markers7[i], color=colors7[i],
                    linewidth=2, label=methods[i])
            # Inline value labels: skip the crowded 0%/1% points everywhere
            # (see corner table below). Additionally, EAP-EDHOC Method 0-3
            # track each other so closely at nearly every loss level that
            # labeling all four individually just overlaps into a smear —
            # for those, only label their most-divergent point (30% loss).
            # Hybrid / EAP-TLS Classic / EAP-TLS Hybrid PQC clearly separate
            # from each other, so they keep full inline labels.
            is_tight_cluster = i in (0, 1, 2, 3)
            stagger = (idx % 4) * 7
            for lx, ly in zip(loss, data[i]):
                if lx in CROWDED_LOSS:
                    continue
                if is_tight_cluster and lx != 30:
                    continue
                ax.annotate(
                    fmt_val(ly),
                    xy=(lx, ly),
                    xytext=(0, 6 + stagger),
                    textcoords="offset points",
                    ha='center',
                    fontsize=6.5,
                    color=colors7[i]
                )
        ax.set_xlabel("Packet Loss (%)")
        ax.set_title(subtitle, fontsize=12, fontweight='bold')
        ax.set_xticks(loss)
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Corner table with the crowded 0%/1% values for all 7 methods
        table_text = build_crowded_table(data)
        ax.text(0.02, 0.97, table_text, transform=ax.transAxes, fontsize=6.5,
                 ha='left', va='top', family='monospace',
                 bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.92))

    # sharey=True ties both panels to the same y-range, so compute headroom
    # from the OVERALL max across both datasets (not per-axis), then apply
    # once via axes[0] (propagates to axes[1] automatically).
    overall_max = max(max(v) for v in list(sim_data.values()) + list(e2e_data.values()))
    axes[0].set_ylim(bottom=min(0, axes[0].get_ylim()[0]), top=overall_max * 1.15)

    axes[0].set_ylabel(ylabel)
    # legend built from the right (end-to-end) panel since it has the
    # full method set (0..6)
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=7, frameon=False,
               bbox_to_anchor=(0.5, 1.06), fontsize=9)
    fig.suptitle(suptitle, fontsize=15, fontweight='bold', y=1.16)

    plt.tight_layout()
    plt.savefig(fname, dpi=600, bbox_inches='tight')
    plt.close()


# ----- Chart 3: Mean latency (gabungan) -----
plot_line_pair(
    sim_mean, e2e_mean,
    ylabel="Mean Handshake Latency (ms)",
    suptitle="Mean Handshake Latency vs Packet Loss: Network-Emulated vs End-to-End",
    fname="combined_mean_latency_loss.png"
)

# ----- Chart 4: p95 latency (gabungan) -----
plot_line_pair(
    sim_p95, e2e_p95,
    ylabel="95th Percentile Latency (ms)",
    suptitle="95th Percentile Handshake Latency vs Packet Loss: Network-Emulated vs End-to-End",
    fname="combined_95th_percentile_latency_loss.png"
)

# ----- Chart 5: Mean retransmissions (gabungan) -----
plot_line_pair(
    sim_retx, e2e_retx,
    ylabel="Mean Retransmissions",
    suptitle="Mean Retransmissions vs Packet Loss: Network-Emulated vs End-to-End",
    fname="combined_mean_retransmissions.png"
)

# =====================================================================
# Chart 6: Success rate heatmap (gabungan, skala warna disamakan)
# Panel kiri tetap Method 0-4 (tidak ada data analitis untuk Method 5/6);
# panel kanan sekarang Method 0-6.
# =====================================================================
loss_labels = ["0%", "1%", "5%", "10%", "20%", "30%"]

sim_methods_keys = sorted(sim_success.keys())
e2e_methods_keys = sorted(e2e_success.keys())

sim_success_arr = np.array([sim_success[k] for k in sim_methods_keys])
e2e_success_arr = np.array([e2e_success[k] for k in e2e_methods_keys])

sim_labels = [methods[k] for k in sim_methods_keys]
e2e_labels = [methods[k] for k in e2e_methods_keys]

vmin_shared = min(sim_success_arr.min(), e2e_success_arr.min())

fig, axes = plt.subplots(1, 2, figsize=(17, 5.2), gridspec_kw={'wspace': 0.25})

for ax, data, labels, subtitle in zip(
    axes, [sim_success_arr, e2e_success_arr], [sim_labels, e2e_labels],
    ["Network-Emulated (Analytical)", "End-to-End (Live EAP-over-RADIUS)"]
):
    im = ax.imshow(data, cmap="Blues_r", vmin=vmin_shared, vmax=100, aspect="auto")
    ax.set_xticks(np.arange(len(loss_labels)))
    ax.set_xticklabels(loss_labels)
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("Packet Loss (%)")
    ax.set_title(subtitle, fontsize=12, fontweight='bold')

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            color = "white" if data[i, j] < (vmin_shared + 100) / 2 else "black"
            ax.text(j, i, f"{data[i, j]:.1f}", ha="center", va="center",
                     fontsize=9, color=color)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel("All Variant Authentication Method")

cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Handshake Success Rate (%)")

fig.suptitle("Handshake Success Rate under Packet Loss: Network-Emulated vs End-to-End",
             fontsize=15, fontweight='bold', y=1.06)

plt.savefig("combined_success_rate_heatmap.png", dpi=600, bbox_inches="tight")
plt.close()

print("All charts generated: chart1, chart2 (with MTU fragmentation), and combined chart3-6 (Method 0-6) with value labels on line charts.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =====================================================================
# Data dari section 1a (identik di analysis.md & analysis_e2e.md)
# n=3000 iterasi per primitif (lihat kolom "Iter" pada tabel sumber —
# BUKAN 10000; n=10000 itu milik uji stabilitas method 0-3 yang terpisah
# di analysis_e2e.md, bukan primitive latency ini)
# =====================================================================
ops = [
    "Keygen_X25519", "ScalarMult_X25519", "Keygen_Ed25519",
    "Signature_Ed25519", "Verify_Ed25519", "SHA256", "HKDF_SHA256",
    "AES128GCM", "Keygen_MLKEM768", "Encaps_MLKEM768", "Decaps_MLKEM768",
    "Keygen_MLDSA44", "Signature_MLDSA44", "Verify_MLDSA44"
]
mean_ns = [
    25535.0, 25163.6, 10730.4, 11069.9, 29438.8, 299.6, 1455.6, 226.0, 32440.5, 33775.5, 41870.0, 56853.6, 223240.5, 59596.1
]
stddev_ns = [
    29916.4, 19026.1, 17311.9, 10090.7, 20344.7, 1233.2, 5021.7, 20.7, 24175.8, 19317.3, 21982.2, 41449.2, 212971.3, 28309.9
]
n = 3000

# Konversi ns -> ms (1 ms = 1,000,000 ns) supaya satuan konsisten dengan chart lain
mean_arr = np.array(mean_ns) / 1e6
stddev_arr = np.array(stddev_ns) / 1e6
ci95 = 1.96 * stddev_arr / np.sqrt(n)  # margin of error 95%, dalam ms

# Kelompok warna: klasik (Ed25519/X25519/SHA/HKDF/AES) vs PQC (ML-KEM/ML-DSA)
is_pqc = [False, False, False, False, False, False, False, False,
          True, True, True, True, True, True]
colors = ["#1565C0" if pqc else "#90CAF9" for pqc in is_pqc]

# Urutkan berdasarkan mean supaya enak dibaca
order = np.argsort(mean_arr)
ops_sorted = [ops[i] for i in order]
mean_sorted = mean_arr[order]
ci_sorted = ci95[order]
colors_sorted = [colors[i] for i in order]

fig, ax = plt.subplots(figsize=(9, 7))
y = np.arange(len(ops_sorted))

ax.barh(y, mean_sorted, xerr=ci_sorted, color=colors_sorted,
        edgecolor='black', linewidth=0.5, capsize=4,
        error_kw=dict(ecolor='black', elinewidth=1.2, capthick=1.2))

ax.set_yticks(y)
ax.set_yticklabels(ops_sorted)
ax.set_xscale('log')
ax.set_xlabel("Latency (ms, log scale) — mean ± 95% CI")
ax.set_title(f"Cryptographic Primitive Latency with 95% Confidence Interval\n(n = {n} iterations per primitive)",
              fontsize=13, fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#90CAF9", edgecolor='black', label='Classical (Ed25519 / X25519 / SHA / HKDF / AES)'),
    Patch(facecolor="#1565C0", edgecolor='black', label='Post-Quantum (ML-KEM-768 / ML-DSA-44)')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=False, fontsize=9)

def fmt_ms(v):
    # Presisi adaptif: nilai kecil (misal SHA256 ~0.00026 ms) butuh lebih banyak desimal
    if v < 0.01:
        return f"{v:.6f}"
    elif v < 1:
        return f"{v:.4f}"
    else:
        return f"{v:,.3f}"

for i, (m, c) in enumerate(zip(mean_sorted, ci_sorted)):
    ax.text(m * 1.15 if m > 0 else 1e-4, i, f"{fmt_ms(m)} ± {fmt_ms(c)} ms",
            va='center', fontsize=8)

plt.tight_layout()
plt.savefig("primitive_latency_confidence_interval.png", dpi=600, bbox_inches='tight')
plt.close()

# =====================================================================
# Tabel CI (untuk dilampirkan di paper)
# =====================================================================
print(f"{'Operation':<20}{'Mean (ms)':>14}{'StdDev (ms)':>14}{'95% CI (±ms)':>16}{'CI Lower':>14}{'CI Upper':>14}")
for op, m_ns, sd_ns in zip(ops, mean_ns, stddev_ns):
    m = m_ns / 1e6
    sd = sd_ns / 1e6
    ci = 1.96 * sd / np.sqrt(n)
    print(f"{op:<20}{m:>14,.6f}{sd:>14,.6f}{ci:>16,.6f}{m-ci:>14,.6f}{m+ci:>14,.6f}")

In [ ]:
"""
analyze_key_bootstrapping.py

Analysis & visualization of the "Key Bootstrapping to Application" phase
(from EMSK): USRK/DSRK derivation (RFC 5295), DSRK transport to the
Application Server, OSCORE context derivation (RFC 8613), and the UE<->AS
first contact.

All timing values are normalized to milliseconds (ms) for consistency
across every chart. All charts use a single blue color family. All labels
are in English so figures can be pasted directly into the paper.

Sections:
    A1 - Primitive operation time (mean only, ms)
    A2 - Compute latency breakdown per component (loss = 0)
    A3 - End-to-end percentile latency vs packet loss (small multiples)
    B1 - Per-leg reliability (retransmissions & latency) vs packet loss
    C  - Key-derivation throughput (compute-only upper bound)
    D  - Baseline comparison (bootstrap vs handshake vs manual login)

Output: all tables (CSV) and figures (PNG) are written to ./output/
"""

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

OUTDIR = "output"
os.makedirs(OUTDIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# ---------------------------------------------------------------------------
# Single blue color family (used consistently across every chart)
# ---------------------------------------------------------------------------
BLUE_DARKEST = "#08306b"
BLUE_DARK    = "#2171b5"
BLUE_MID     = "#4292c6"
BLUE         = "#6baed6"
BLUE_LIGHT   = "#9ecae1"
BLUE_LIGHTEST = "#c6dbef"
BLUE_PALETTE = [BLUE_DARKEST, BLUE_DARK, BLUE_MID, BLUE, BLUE_LIGHT, BLUE_LIGHTEST]


# ---------------------------------------------------------------------------
# 1. DATA (source tables from the bootstrap benchmark report)
#    All raw values are converted to milliseconds (ms) right here.
# ---------------------------------------------------------------------------

# --- A1. Primitive derivation time (source unit: ns -> converted to ms) ---
df_a1 = pd.DataFrame([
    {"operation": "USRK Derivation (RFC 5295)",        "mean_ns": 2658.3},
    {"operation": "DSRK Derivation (RFC 5295)",         "mean_ns": 2483.6},
    {"operation": "OSCORE Master Secret Derivation",    "mean_ns": 1281.9},
    {"operation": "OSCORE Master Salt Derivation",      "mean_ns": 1219.1},
    {"operation": "OSCORE Context Derivation (RFC 8613)","mean_ns": 6916.8},
    {"operation": "OSCORE Protect (AES-CCM)",           "mean_ns": 7584.3},
    {"operation": "OSCORE Unprotect (AES-CCM)",         "mean_ns": 7367.8},
])
df_a1["mean_ms"] = df_a1["mean_ns"] / 1e6

# --- A2. Compute latency breakdown per component (loss = 0, source: us -> ms) ---
df_a2 = pd.DataFrame([
    {"component": "USRK+DSRK Derivation (UE)",      "mean_us": 5.540},
    {"component": "USRK+DSRK Derivation (AAA)",     "mean_us": 6.944},
    {"component": "OSCORE Context Derivation (UE)", "mean_us": 9.831},
    {"component": "OSCORE Context Derivation (AS)", "mean_us": 11.388},
])
df_a2["mean_ms"] = df_a2["mean_us"] / 1000

E2E_BOOTSTRAP_MEAN_MS = 0.4183
E2E_BOOTSTRAP_P95_MS = 0.4253

# --- A3. End-to-end percentile latency vs packet loss (already ms) ---
df_a3 = pd.DataFrame([
    {"loss_pct": 0.0  , "mean_ms": 0.4176, "p50_ms": 0.4158, "p95_ms": 0.4243, "p99_ms": 0.4668},
    {"loss_pct": 1.0  , "mean_ms": 1.0578, "p50_ms": 0.4158, "p95_ms": 0.4266, "p99_ms": 40.4157},
    {"loss_pct": 5.0  , "mean_ms": 4.578, "p50_ms": 0.4157, "p95_ms": 40.4157, "p99_ms": 80.4154},
    {"loss_pct": 10.0 , "mean_ms": 8.8981, "p50_ms": 0.4158, "p95_ms": 40.4164, "p99_ms": 80.4157},
    {"loss_pct": 20.0 , "mean_ms": 19.6177, "p50_ms": 0.416, "p95_ms": 80.4159, "p99_ms": 120.4157},
    {"loss_pct": 30.0 , "mean_ms": 34.1776, "p50_ms": 40.4153, "p95_ms": 120.4157, "p99_ms": 160.4166},
])

# --- B1. Per-leg reliability vs packet loss (already ms) ---
df_b1 = pd.DataFrame([
    {"leg": "AAA->AS", "loss_pct": 0.0  , "mean_ms": 0.2, "mean_retx": 0},
    {"leg": "UE->AS", "loss_pct": 0.0  , "mean_ms": 0.2, "mean_retx": 0},
    {"leg": "AAA->AS", "loss_pct": 1.0  , "mean_ms": 0.48, "mean_retx": 0.007},
    {"leg": "UE->AS", "loss_pct": 1.0  , "mean_ms": 0.56, "mean_retx": 0.009},
    {"leg": "AAA->AS", "loss_pct": 5.0  , "mean_ms": 2.56, "mean_retx": 0.059},
    {"leg": "UE->AS", "loss_pct": 5.0  , "mean_ms": 2.0, "mean_retx": 0.045},
    {"leg": "AAA->AS", "loss_pct": 10.0 , "mean_ms": 4.44, "mean_retx": 0.106},
    {"leg": "UE->AS", "loss_pct": 10.0 , "mean_ms": 4.44, "mean_retx": 0.106},
    {"leg": "AAA->AS", "loss_pct": 20.0 , "mean_ms": 9.96, "mean_retx": 0.244},
    {"leg": "UE->AS", "loss_pct": 20.0 , "mean_ms": 9.64, "mean_retx": 0.236},
    {"leg": "AAA->AS", "loss_pct": 30.0 , "mean_ms": 17.04, "mean_retx": 0.421},
    {"leg": "UE->AS", "loss_pct": 30.0 , "mean_ms": 17.12, "mean_retx": 0.423},
])

# --- B2. Key consistency rate -- kept as a plain number, no chart (per request) ---
KEY_CONSISTENCY = {"sessions": 6000, "consistent": 6000}

# --- C. System load & scalability (compute-only, single-core, source: us -> ms) ---
df_c = pd.DataFrame([
    {"component": "AAA\n(USRK + DSRK)",                    "time_per_session_us": 5.142, "throughput_sessions_per_sec_core": 194481},
    {"component": "AS\n(OSCORE secret/salt + context)",    "time_per_session_us": 9.418, "throughput_sessions_per_sec_core": 106182},
])
df_c["time_per_session_ms"] = df_c["time_per_session_us"] / 1000

STORAGE_OVERHEAD = [
    {"side": "AAA", "state": "DSRK per UE per domain", "size_bytes": 64},
    {"side": "AS",  "state": "OSCORE context (secret/salt + sender/recipient key + IV)", "size_bytes": 71},
]

# --- D. UX baseline comparison (already ms) ---
BOOTSTRAP_LATENCY_MS = 0.4183
OAUTH_LOWER_BOUND_MS = 2000.0    # literature estimate, NOT measured
OAUTH_UPPER_BOUND_MS = 10000.0   # literature estimate, NOT measured
EDHOC_HANDSHAKE_MEAN_MS = 14.733  # from analisys2.md, EAP-EDHOC Hybrid end-to-end


# ---------------------------------------------------------------------------
# 2. Optional loader for raw per-session data (CSV/JSON), if available
# ---------------------------------------------------------------------------

def load_raw_if_available(csv_or_json_path: str):
    """Load raw benchmark data from file if it exists, else return None."""
    if not csv_or_json_path or not os.path.exists(csv_or_json_path):
        return None
    if csv_or_json_path.endswith(".csv"):
        return pd.read_csv(csv_or_json_path)
    if csv_or_json_path.endswith(".json"):
        with open(csv_or_json_path) as f:
            return pd.DataFrame(json.load(f))
    raise ValueError("Unrecognized file format, use .csv or .json")


# ---------------------------------------------------------------------------
# 3. Visualizations
# ---------------------------------------------------------------------------

def plot_a1_primitive_mean(df: pd.DataFrame, outpath: str):
    """A1 - Mean primitive operation time only (ms), single blue tone."""
    df_sorted = df.sort_values("mean_ms", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(df_sorted["operation"], df_sorted["mean_ms"], color=BLUE_DARK)
    for b, v in zip(bars, df_sorted["mean_ms"]):
        ax.text(v, b.get_y() + b.get_height() / 2, f"  {v:.4f} ms", va="center", fontsize=9)
    ax.set_xlabel("Mean Time (ms)")
    ax.set_title("Primitive Derivation Time (Mean) — Key Bootstrapping")
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def plot_a2_compute_breakdown(df: pd.DataFrame, e2e_mean_ms: float, outpath: str):
    """A2 - Compute breakdown vs network overhead (ms), blue tones."""
    fig, ax = plt.subplots(figsize=(8, 5))
    compute_total_ms = df["mean_ms"].sum()
    network_ms = e2e_mean_ms - compute_total_ms

    labels = list(df["component"]) + ["Network Overhead (2 legs)"]
    values = list(df["mean_ms"]) + [network_ms]
    colors = [BLUE_LIGHT, BLUE, BLUE_MID, BLUE_DARK, BLUE_DARKEST][: len(labels)]

    ax.barh(labels, values, color=colors)
    ax.set_xlabel("Time (ms)")
    ax.set_title(
        f"Bootstrap Latency Breakdown (loss = 0)\n"
        f"Total end-to-end mean = {e2e_mean_ms:.4f} ms "
        f"(compute = {compute_total_ms:.4f} ms, network = {network_ms:.4f} ms)"
    )
    for i, v in enumerate(values):
        ax.text(v, i, f"  {v:.4f} ms", va="center", fontsize=9)
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def plot_a3_percentile_vs_loss(df: pd.DataFrame, outpath: str):
    """Redesigned as small multiples (one subplot per metric) for clarity.
    Each subplot uses a linear scale in ms, avoiding overlapping/log clutter."""
    metrics = [
        ("mean_ms", "Mean Latency"),
        ("p50_ms", "p50 Latency (Median)"),
        ("p95_ms", "p95 Latency"),
        ("p99_ms", "p99 Latency"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
    axes = axes.flatten()

    for ax, (col, title) in zip(axes, metrics):
        ax.plot(df["loss_pct"], df[col], "-o", color=BLUE_DARK, linewidth=2, markersize=6)
        ax.set_title(title)
        ax.set_xlabel("Packet Loss (%)")
        ax.set_ylabel("Latency (ms)")
        for x, y in zip(df["loss_pct"], df[col]):
            ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points",
                        xytext=(0, 6), fontsize=7, ha="center", color=BLUE_DARKEST)

    fig.suptitle(
        "End-to-End Bootstrap Latency vs Packet Loss (Key Bootstrapping)\n"
        "Note: discrete jumps in p95/p99 come from the fixed 40 ms retransmission timeout (RTO)",
        fontsize=11,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(outpath)
    plt.close(fig)


def plot_b1_reliability_per_leg(df: pd.DataFrame, outpath: str):
    """Per-leg reliability (retransmissions & latency) vs packet loss, blue tones.
    Value labels are added at every point >=5% loss, offset diagonally per
    leg (AAA->AS upper-left, UE->AS lower-right) so the two legs — whose
    values are often nearly identical — don't collide. The 0% and 1%
    points sit only 1 unit apart on a 0-30 axis and their values are
    almost identical between legs, so inline labels there just overlap;
    those four numbers are instead summarized in a small text box."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    leg_colors = {"AAA->AS": BLUE_DARK, "UE->AS": BLUE_LIGHT}
    leg_markers = {"AAA->AS": "o", "UE->AS": "s"}
    leg_label_offset = {"AAA->AS": (-16, 9), "UE->AS": (16, -13)}
    leg_ha = {"AAA->AS": "right", "UE->AS": "left"}
    crowded_loss = {0.0, 1.0}

    def build_crowded_box(sub_by_leg, fmt):
        lines = []
        for lp in sorted(crowded_loss):
            parts = [f"{leg}={fmt(sub_by_leg[leg].loc[sub_by_leg[leg]['loss_pct'] == lp].iloc[0])}"
                     for leg in ["AAA->AS", "UE->AS"]]
            lines.append(f"{lp:.0f}%: " + ", ".join(parts))
        return "\n".join(lines)

    # --- Retransmissions panel ---
    sub_by_leg = {leg: sub for leg, sub in df.groupby("leg")}
    for leg, sub in sub_by_leg.items():
        axes[0].plot(sub["loss_pct"], sub["mean_retx"], marker=leg_markers[leg],
                     color=leg_colors[leg], linewidth=2, label=leg)
        dx, dy = leg_label_offset[leg]
        for x, y in zip(sub["loss_pct"], sub["mean_retx"]):
            if x in crowded_loss:
                continue
            axes[0].annotate(f"{y:.4f}", (x, y), textcoords="offset points",
                              xytext=(dx, dy), fontsize=7,
                              ha=leg_ha[leg], color=leg_colors[leg])
    box_text = build_crowded_box({leg: sub[["loss_pct", "mean_retx"]].rename(columns={"mean_retx": "v"})
                                   for leg, sub in sub_by_leg.items()}, lambda row: f"{row['v']:.4f}")
    axes[0].text(0.98, 0.03, box_text, transform=axes[0].transAxes, fontsize=7,
                 ha="right", va="bottom",
                 bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="gray", alpha=0.9))
    axes[0].set_xlabel("Packet Loss (%)")
    axes[0].set_ylabel("Mean Retransmissions")
    axes[0].set_title("Mean Retransmissions vs Packet Loss (per leg)")
    axes[0].legend()
    axes[0].margins(x=0.08, y=0.2)

    # --- Latency panel ---
    for leg, sub in sub_by_leg.items():
        axes[1].plot(sub["loss_pct"], sub["mean_ms"], marker=leg_markers[leg],
                     color=leg_colors[leg], linewidth=2, label=leg)
        dx, dy = leg_label_offset[leg]
        for x, y in zip(sub["loss_pct"], sub["mean_ms"]):
            if x in crowded_loss:
                continue
            axes[1].annotate(f"{y:.2f}", (x, y), textcoords="offset points",
                              xytext=(dx, dy), fontsize=7,
                              ha=leg_ha[leg], color=leg_colors[leg])
    box_text = build_crowded_box({leg: sub[["loss_pct", "mean_ms"]].rename(columns={"mean_ms": "v"})
                                   for leg, sub in sub_by_leg.items()}, lambda row: f"{row['v']:.2f} ms")
    axes[1].text(0.98, 0.03, box_text, transform=axes[1].transAxes, fontsize=7,
                 ha="right", va="bottom",
                 bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="gray", alpha=0.9))
    axes[1].set_xlabel("Packet Loss (%)")
    axes[1].set_ylabel("Mean Latency (ms)")
    axes[1].set_title("Mean Latency vs Packet Loss (per leg)")
    axes[1].legend()
    axes[1].margins(x=0.08, y=0.2)

    fig.suptitle("Per-Leg Reliability (AAA→AS vs UE→AS) — success rate 100% at every point")
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def plot_c_throughput(df: pd.DataFrame, outpath: str):
    """Key derivation throughput, compute-only upper bound, blue tones."""
    fig, ax = plt.subplots(figsize=(7, 5))
    colors = [BLUE_DARK, BLUE_MID]
    bars = ax.bar(df["component"], df["throughput_sessions_per_sec_core"], color=colors)
    ax.set_ylabel("Sessions/sec/core (compute-only upper bound)")
    ax.set_title("Key Derivation Throughput (Theoretical Upper Bound)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
    for b, v, t in zip(bars, df["throughput_sessions_per_sec_core"], df["time_per_session_ms"]):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,.0f}/s\n({t:.4f} ms/session)",
                ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


def plot_d_baseline_comparison(outpath: str):
    """Baseline comparison, all bars in one blue tone family, log scale in ms."""
    fig, ax = plt.subplots(figsize=(7, 5))
    labels = [
        "Key Bootstrapping\n(measured)",
        "EAP-EDHOC Handshake\n(measured, EAP-EDHOC Hybrid e2e)",
        "Manual OAuth Login",
    ]
    values_ms = [
        BOOTSTRAP_LATENCY_MS,
        EDHOC_HANDSHAKE_MEAN_MS,
        (OAUTH_LOWER_BOUND_MS + OAUTH_UPPER_BOUND_MS) / 2,
    ]
    colors = [BLUE_LIGHT, BLUE_MID, BLUE_DARKEST]

    bars = ax.bar(labels, values_ms, color=colors)
    ax.set_yscale("log")
    ax.set_ylabel("Latency (ms, log scale)")
    ax.set_title(
        "Latency Comparison: Bootstrapping vs Handshake vs Manual Login\n"
        "OAuth baseline"
    )

    ax.errorbar(
        [2], [values_ms[2]],
        yerr=[[values_ms[2] - OAUTH_LOWER_BOUND_MS], [OAUTH_UPPER_BOUND_MS - values_ms[2]]],
        fmt="none", ecolor=BLUE_DARKEST, capsize=6,
    )
    for b, v in zip(bars, values_ms):
        ax.text(b.get_x() + b.get_width() / 2, v * 1.15, f"{v:,.2f} ms", ha="center", fontsize=9)
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 4. Key summary numbers (for direct citation in the paper)
# ---------------------------------------------------------------------------

def print_summary():
    compute_total_ms = df_a2["mean_ms"].sum()
    network_ms = E2E_BOOTSTRAP_MEAN_MS - compute_total_ms
    pct_of_handshake = E2E_BOOTSTRAP_MEAN_MS / EDHOC_HANDSHAKE_MEAN_MS * 100
    speedup_low = OAUTH_LOWER_BOUND_MS / BOOTSTRAP_LATENCY_MS

    lines = [
        "=== KEY SUMMARY NUMBERS (Key Bootstrapping) — all times in ms ===",
        f"- End-to-end bootstrap latency (loss 0): mean={E2E_BOOTSTRAP_MEAN_MS} ms, p95={E2E_BOOTSTRAP_P95_MS} ms",
        f"- Total compute time (4 components): {compute_total_ms:.4f} ms",
        f"- Estimated network overhead (2 legs): {network_ms:.4f} ms",
        f"- Overhead relative to main EAP-EDHOC handshake ({EDHOC_HANDSHAKE_MEAN_MS} ms): {pct_of_handshake:.2f}%",
        f"- Key consistency: {KEY_CONSISTENCY['consistent']}/{KEY_CONSISTENCY['sessions']} "
        f"({KEY_CONSISTENCY['consistent']/KEY_CONSISTENCY['sessions']*100:.1f}%)",
        f"- Speedup vs lower-bound OAuth manual-login estimate (2000 ms): ~{speedup_low:,.0f}x",
        f"- Success rate: 100% at all packet-loss points (0-30%) on both legs",
        f"- Compute-only throughput: AAA={df_c.iloc[0]['throughput_sessions_per_sec_core']:,.0f} sessions/s/core, "
        f"AS={df_c.iloc[1]['throughput_sessions_per_sec_core']:,.0f} sessions/s/core",
    ]
    text = "\n".join(lines)
    print(text)
    with open(os.path.join(OUTDIR, "summary.txt"), "w") as f:
        f.write(text + "\n")


# ---------------------------------------------------------------------------
# 5. Main
# ---------------------------------------------------------------------------

def main():
    # Save raw tables as CSV (ms-normalized) for reuse in LaTeX/Excel
    df_a1[["operation", "mean_ms"]].to_csv(os.path.join(OUTDIR, "A1_primitive_mean_ms.csv"), index=False)
    df_a2[["component", "mean_ms"]].to_csv(os.path.join(OUTDIR, "A2_compute_breakdown_ms.csv"), index=False)
    df_a3.to_csv(os.path.join(OUTDIR, "A3_percentile_vs_loss_ms.csv"), index=False)
    df_b1.to_csv(os.path.join(OUTDIR, "B1_reliability_per_leg_ms.csv"), index=False)
    df_c[["component", "time_per_session_ms", "throughput_sessions_per_sec_core"]].to_csv(
        os.path.join(OUTDIR, "C_throughput_ms.csv"), index=False)
    pd.DataFrame(STORAGE_OVERHEAD).to_csv(os.path.join(OUTDIR, "C_storage_overhead.csv"), index=False)

    # Figures
    plot_a1_primitive_mean(df_a1, os.path.join(OUTDIR, "A1_primitive_mean.png"))
    plot_a2_compute_breakdown(df_a2, E2E_BOOTSTRAP_MEAN_MS, os.path.join(OUTDIR, "A2_compute_breakdown.png"))
    plot_a3_percentile_vs_loss(df_a3, os.path.join(OUTDIR, "A3_percentile_vs_loss.png"))
    plot_b1_reliability_per_leg(df_b1, os.path.join(OUTDIR, "B1_reliability_per_leg.png"))
    plot_c_throughput(df_c, os.path.join(OUTDIR, "C_throughput.png"))
    plot_d_baseline_comparison(os.path.join(OUTDIR, "D_baseline_comparison.png"))

    print_summary()
    print(f"\nAll tables & figures saved to: {os.path.abspath(OUTDIR)}")


if __name__ == "__main__":
    main()